# 03 · Feature experiments

Features that were tried and are **not** in the pipeline, with the evidence. The
plan's optional `recent_late_rate_14d` is tested here in two versions.

Run after `make features` and `make train`: it reads the warehouse and reuses the
final model's parameters from `artifacts/model_bundle.joblib`. Like model
selection, it uses only the three folds and validation; the test set is not
touched. Every fit only learns from outcomes known when its period starts.

In [1]:
import duckdb
import pandas as pd
from sklearn.metrics import roc_auc_score

from opspilot.config import DUCKDB_PATH
from opspilot.features.load import FEATURES, TARGET, load_features
from opspilot.models.baselines import logistic_regression_on
from opspilot.models.bundle import load_bundle
from opspilot.models.selection import evaluate_periods, predict_on_periods, selection_summary
from opspilot.models.train import xgboost_fit_predict

pd.set_option("display.float_format", "{:,.3f}".format)

/Users/incilaldikmen/OpsPilot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Two ways to measure the recent late rate

Both are the late rate among outcomes that arrived in the 14 days before t0;
they differ in **when** an outcome counts as arrived.

- **Delivered** (the plan's definition): when the order was delivered. A late
  order is the slow one, so it only counts weeks after it went wrong.
- **Known**: when the outcome became known (`fct_orders.outcome_known_at`). A late
  order counts the day after its promised date passes, delivered or not.

Running totals plus an as-of join, so each order sees only events before its t0.

In [2]:
RECENT_SQL = """
with events as (
    select {event} as event_at, count(*) as n, sum(is_late) as late
    from fct_orders
    where is_labeled
    group by 1
),
running as (
    select event_at, sum(n) over w as n, sum(late) over w as late
    from events
    window w as (order by event_at)
),
orders as (select order_id, t0 from ml_late_delivery_features),
at_t0 as (
    select orders.order_id, running.n, running.late
    from orders asof left join running on orders.t0 > running.event_at
),
at_start as (
    select orders.order_id, running.n, running.late
    from orders asof left join running on orders.t0 - interval 14 days > running.event_at
)
select
    at_t0.order_id,
    (coalesce(at_t0.late, 0) - coalesce(at_start.late, 0))
    / nullif(coalesce(at_t0.n, 0) - coalesce(at_start.n, 0), 0) as {name}
from at_t0
inner join at_start using (order_id)
"""

con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
recent = con.sql(RECENT_SQL.format(event="delivered_at", name="recent_late_rate_delivered")).df()
recent = recent.merge(
    con.sql(RECENT_SQL.format(event="outcome_known_at", name="recent_late_rate_known")).df(),
    on="order_id",
)
con.close()

features = load_features().merge(recent, on="order_id", how="left")
VARIANTS = ["recent_late_rate_delivered", "recent_late_rate_known"]
features[VARIANTS].describe().T

,count,mean,std,min,25%,50%,75%,max
recent_late_rate_delivered,"86,013.000",0.067,0.052,0.000,0.031,0.048,0.094,0.333
recent_late_rate_known,"86,251.000",0.069,0.074,0.000,0.031,0.045,0.080,1.000


## What each version sees during the spike

Validation covers January to March 2018, when the late rate jumped from 5.7% to
19%. The feature is only useful if it rises with it.

In [3]:
spike = features[features.purchase_date.between("2018-01-01", "2018-03-31")].copy()
spike["month"] = pd.to_datetime(spike.purchase_date).dt.strftime("%Y-%m")
spike.groupby("month").agg(
    actual_late_rate=(TARGET, "mean"),
    delivered_version=("recent_late_rate_delivered", "mean"),
    known_version=("recent_late_rate_known", "mean"),
)

,actual_late_rate,delivered_version,known_version
month,,,
2018-01,0.057,0.127,0.089
2018-02,0.141,0.046,0.058
2018-03,0.190,0.102,0.143


In [4]:
periods = {
    "train": ("2017-01-01", "2017-12-31"),
    "validation": ("2018-01-01", "2018-03-31"),
}
pd.DataFrame(
    {
        name: {
            variant: roc_auc_score(part.loc[part[variant].notna(), TARGET], part[variant].dropna())
            for variant in VARIANTS
        }
        for name, (start, end) in periods.items()
        for part in [features[features.purchase_date.between(start, end)]]
    }
).rename_axis("ROC-AUC on its own")

,train,validation
ROC-AUC on its own,,
recent_late_rate_delivered,0.560,0.437
recent_late_rate_known,0.555,0.497


## With and without, on the folds and validation

Logistic regression and the final XGBoost parameters, each on the agreed 27
features and with one version of the recent rate added.

In [5]:
params = load_bundle()["params"]
rows = []
for added in [None, *VARIANTS]:
    feature_list = FEATURES + ([added] if added else [])
    for model, fit_predict in [
        ("logistic regression", logistic_regression_on(feature_list)),
        ("xgboost", xgboost_fit_predict(params, feature_list)),
    ]:
        results = evaluate_periods(predict_on_periods(features, fit_predict))
        rows.append(
            {
                "model": model,
                "added feature": added or "(none)",
                **{period: r["lift"] for period, r in results.items()},
                **selection_summary(results),
            }
        )
experiments = pd.DataFrame(rows).set_index(["model", "added feature"])
experiments[["fold_1", "fold_2", "fold_3", "validation", "selection_lift", "selection_roc_auc"]]

,,fold_1,fold_2,fold_3,validation,selection_lift,selection_roc_auc
model,added feature,,,,,,
logistic regression,(none),1.515,2.209,1.572,2.071,1.842,0.662
xgboost,(none),1.927,2.668,1.660,2.236,2.123,0.683
logistic regression,recent_late_rate_delivered,1.504,2.047,1.578,1.494,1.656,0.638
xgboost,recent_late_rate_delivered,1.446,2.621,1.667,2.146,1.970,0.669
logistic regression,recent_late_rate_known,1.482,2.137,1.553,2.008,1.795,0.656
xgboost,recent_late_rate_known,1.622,2.514,1.633,2.152,1.980,0.669


## Conclusion

Neither version goes in the feature table.

- **Delivered** lags the spike. In February 2018 it read 4.6% while that month's
  orders ran 14.1% late, and inside validation it is anti-correlated with the
  label (ROC-AUC 0.44 on its own).
- **Known** removes the anti-correlation (0.50) and tracks March better (14.3%
  against 19.0%), but still trails the jump in February (5.8% against 14.1%). A
  broken promise only shows once the promised day has passed, about three weeks
  after purchase for the median order.
- Added to either model, both versions lower the mean lift: XGBoost from 2.09 to
  2.02 (delivered) and 1.98 (known), logistic regression from 1.82 to 1.65 and
  1.79.

A spike begins with orders whose promised day is still ahead, so any feature
built from outcomes arrives too late for the orders that matter. Seeing a spike
coming would take leading signals this dataset does not have, such as carrier
capacity or weather.